# Paso 1

#### Librerias y lectura

In [55]:
import json
import re
import pandas as pd
import unicodedata
from thefuzz import fuzz

In [56]:
df = pd.read_json('catalogo_tp3.jsonl', lines=True)

#### Vista inicial

In [57]:
total_inicial = len(df)
print(f"Total de registros iniciales: {total_inicial}")

Total de registros iniciales: 8747


In [58]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   source_url      8747 non-null   str                
 1   slug            8747 non-null   str                
 2   title           8747 non-null   str                
 3   authors         8747 non-null   object             
 4   genres          8747 non-null   object             
 5   seed_genre      8747 non-null   str                
 6   series          2316 non-null   str                
 7   series_number   2298 non-null   float64            
 8   cover_url       8747 non-null   str                
 9   synopsis        8746 non-null   str                
 10  synopsis_chars  8747 non-null   int64              
 11  scraped_at      8747 non-null   datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), float64(1), int64(1), object(2), str(7)
memory usage: 820.2+ KB


In [59]:
df.head()

,source_url,slug,title,authors,genres,seed_genre,series,series_number,cover_url,synopsis,synopsis_chars,scraped_at
0,https://ww3.lectulandia.com/book/de-la-caverna...,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eudald%20C...,La Humanidad ha entrado en una nueva fase evol...,998,2026-10-17 23:11:56+00:00
1,https://ww3.lectulandia.com/book/quienes-somos...,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, En...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/David%20Re...,La extracción de ADN de los huesos de nuestros...,1285,2026-10-17 23:11:58+00:00
2,https://ww3.lectulandia.com/book/teotihuacan/,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo,...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eduardo%20...,Teotihuacan es la primera gran ciudad del cent...,1070,2026-10-17 23:12:01+00:00
3,https://ww3.lectulandia.com/book/tiro-y-las-co...,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"[Arqueología, Economía, Ensayo, Historia, Polí...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/M%20Eugeni...,Entre los siglos VIII y VI a. C. los fenicios ...,1875,2026-10-17 23:12:03+00:00
4,https://ww3.lectulandia.com/book/la-maldicion-...,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],"[Arqueología, Ensayo, Psicología, Sociología]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Emiliano%2...,Las raíces evolutivas del sufrimiento humano\n...,1613,2026-10-17 23:12:05+00:00


#### Analisis de sinopsis

In [60]:
(df['synopsis'] == "").sum()

np.int64(0)

No hay sinopsis vacias

In [61]:
# detectar filas con sinapsis = a NaN
df[df['synopsis'].isnull()]

,source_url,slug,title,authors,genres,seed_genre,series,series_number,cover_url,synopsis,synopsis_chars,scraped_at
7969,https://ww3.lectulandia.com/book/calvario/,calvario,Calvario,[Alejandro Sawa],"[Drama, Teatro]",teatro,NaN,NaN,https://assets.lectulandia.com/b/ab/Alejandro%...,NaN,0,2026-10-18 22:48:58+00:00


Hay una sola fila con sinapsis vacia

In [62]:
df = df.dropna(subset=["synopsis"])
df = df[df["synopsis"].astype(str).str.strip() != ""]

In [63]:
# detectar cuantas y eliminar las filas con slug duplicados
df.duplicated(subset=['slug']).sum()

np.int64(0)

No hay libros duplicados

In [64]:
df = df.drop_duplicates(subset=["slug"], keep="first")

In [65]:
len(df)

8746

In [66]:
# Normalizar espacios en blanco (reemplazar saltos de línea y tabs por un único espacio)
antes = len(df)
df["synopsis"] = df["synopsis"].astype(str).apply(lambda x: re.sub(r"\s+", " ", x).strip())
df["title"] = df["title"].astype(str).apply(lambda x: re.sub(r"\s+", " ", x).strip())

In [67]:
# Contar duplicados exactos de sinopsis
sinopsis_duplicadas = df[df.duplicated(subset=["synopsis"], keep=False)]
print(f"Registros con sinopsis iguales: {len(sinopsis_duplicadas)}")

Registros con sinopsis iguales: 131


In [68]:
# Conservar una sola versión por sinopsis
df = df.drop_duplicates(subset=["synopsis"], keep="first")
eliminados_sinopsis = antes - len(df)
print(f"Libros eliminados por sinopsis iguales: {eliminados_sinopsis}")

Libros eliminados por sinopsis iguales: 98


#### Busqueda de libros "casi iguales"

Normalizacion de los titulos

In [69]:
def normalizar_titulo(texto):
    if pd.isna(texto):
        return ""

    texto = texto.lower()

    # Quitar acentos
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    # Eliminar puntuación
    texto = re.sub(r"[^\w\s]", " ", texto)

    # Espacios múltiples
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


df["title_norm"] = df["title"].apply(normalizar_titulo)

Buscar candidatos

In [70]:
# Columna auxiliar para agrupar por autor
df["authors_norm"] = df["authors"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else str(x)
)

pares = []

romanos = r"\b(i|ii|iii|iv|v|vi|vii|viii|ix|x)\b"

for autor, grupo in df.groupby("authors_norm"):

    grupo = grupo.reset_index()

    if len(grupo) < 2:
        continue

    for i in range(len(grupo)):
        for j in range(i + 1, len(grupo)):

            titulo1 = grupo.loc[i, "title_norm"]
            titulo2 = grupo.loc[j, "title_norm"]

            score = fuzz.token_set_ratio(titulo1, titulo2)

            if score < 90:
                continue

            # Si tienen números distintos (años, tomos, etc.)
            nums1 = re.findall(r"\d+", titulo1)
            nums2 = re.findall(r"\d+", titulo2)

            if nums1 != nums2:
                continue

            # Si cambian los números romanos
            rom1 = re.findall(romanos, titulo1)
            rom2 = re.findall(romanos, titulo2)

            if rom1 != rom2:
                continue

            # Evitar tomos o volúmenes diferentes
            palabras = ["vol", "volumen", "tomo", "parte", "libro", "numero"]

            if any(p in titulo1 or p in titulo2 for p in palabras):
                if titulo1 != titulo2:
                    continue

            pares.append({
                "idx1": grupo.loc[i, "index"],
                "idx2": grupo.loc[j, "index"],
                "autor": autor,
                "titulo1": grupo.loc[i, "title"],
                "titulo2": grupo.loc[j, "title"],
                "score_titulo": score
            })

pares = pd.DataFrame(pares)

print(f"Pares candidatos encontrados: {len(pares)}")
display(pares.head())

Pares candidatos encontrados: 110


,idx1,idx2,autor,titulo1,titulo2,score_titulo
0,451,7950,"Adolfo Bioy Casares, Jorge Luis Borges",Nuevos cuentos de Bustos Domecq,Nuevos cuentos de Bustos Domecq,100
1,490,2090,Alejandro Fabbri,Historias negras del fútbol argentino,Nuevas historias negras del fútbol argentino,100
2,529,2857,Ambrose Bierce,El diccionario del Diablo (Trad. Eduardo Stilman),El diccionario del Diablo,100
3,2857,3590,Ambrose Bierce,El diccionario del Diablo,El diccionario del diablo (trad. Rodolfo Walsh),100
4,1603,7955,Anónimo,Lazarillo de Tormes (ed. Francisco Rico),Lazarillo de Tormes,100


Comparar sinopsis

In [71]:
pares["score_sinopsis"] = pares.apply(
    lambda fila: fuzz.token_set_ratio(
        df.loc[fila["idx1"], "synopsis"],
        df.loc[fila["idx2"], "synopsis"]
    ),
    axis=1
)

display(pares.head())

,idx1,idx2,autor,titulo1,titulo2,score_titulo,score_sinopsis
0,451,7950,"Adolfo Bioy Casares, Jorge Luis Borges",Nuevos cuentos de Bustos Domecq,Nuevos cuentos de Bustos Domecq,100,59
1,490,2090,Alejandro Fabbri,Historias negras del fútbol argentino,Nuevas historias negras del fútbol argentino,100,58
2,529,2857,Ambrose Bierce,El diccionario del Diablo (Trad. Eduardo Stilman),El diccionario del Diablo,100,51
3,2857,3590,Ambrose Bierce,El diccionario del Diablo,El diccionario del diablo (trad. Rodolfo Walsh),100,58
4,1603,7955,Anónimo,Lazarillo de Tormes (ed. Francisco Rico),Lazarillo de Tormes,100,59


Elegir las posibles "obras iguales" y mostrarlas

In [72]:
obras_iguales = pares[
    pares["score_sinopsis"] >= 80
].sort_values(
    ["score_titulo", "score_sinopsis"],
    ascending=False
)

print(f"Posibles obras iguales: {len(obras_iguales)}")

Posibles obras iguales: 25


Mostramos estas obras

In [73]:
display(
    obras_iguales[
        [
            "idx1",
            "idx2",
            "autor",
            "titulo1",
            "titulo2",
            "score_titulo",
            "score_sinopsis"
        ]
    ]
)

,idx1,idx2,autor,titulo1,titulo2,score_titulo,score_sinopsis
7,1309,1504,Arthur Conan Doyle,Sherlock Holmes,Sherlock Holmes. La colección completa,100,100
37,4862,5876,H. Rider Haggard,"Ella, Hija de la Sabiduría",Ella. Hija de la sabiduría,100,100
43,206,6001,Isaac Asimov,El secreto del universo,El secreto del universo y otros ensayos cientí...,100,100
53,208,6218,John Gribbin,Introducción a la ciencia. Una guía para todos...,Introducción a la ciencia,100,100
61,1550,1803,Lewis Wallace,Ben-Hur,Ben-Hur (Traducción Raimundo Griñó),100,100
64,2065,8462,Mark Twain,El príncipe y el mendigo,Príncipe y mendigo,100,100
65,736,739,Maurice Nicoll,Comentarios psicológicos sobre las enseñanzas ...,Comentarios psicológicos sobre las enseñanzas ...,100,100
82,8028,8032,Richard Ford,Manual para viajeros por Andalucía y lectores ...,Manual para viajeros por Andalucía y lectores ...,100,100
49,359,7456,Jean-Jacques Rousseau,El contrato social,Contrato social,100,99
60,801,5290,Laura Gallego García,La emperatriz de los Etéreos (Ed. Ilustrada),La emperatriz de los Etéreos,100,99


### Decisión sobre los candidatos a obras iguales

La detección automática produjo 25 pares candidatos a corresponder a una misma obra. Sin embargo, la similitud de títulos y sinopsis no garantiza que dos registros representen el mismo libro.

Tras una revisión manual, solo se eliminaron aquellos casos en los que existía evidencia suficiente de que ambos registros correspondían a la misma obra, diferenciándose únicamente por aspectos editoriales como puntuación, subtítulos, traducciones o la indicación de una edición ilustrada.

En cambio, se conservaron aquellos pares que, aunque presentan una alta similitud textual, representan obras distintas. Entre ellos se encuentran recopilaciones que contienen material adicional (por ejemplo, *Sherlock Holmes. La colección completa* o *El secreto del universo y otros ensayos científicos*), así como libros pertenecientes a una misma colección pero dedicados a regiones diferentes (por ejemplo, las guías de viaje de Richard Ford para Andalucía, Castilla, Cataluña o el País Vasco). Eliminar estos registros implicaría perder información válida del catálogo.

Por este motivo, la decisión final combina la detección automática de candidatos con una validación manual, priorizando la conservación de información cuando existen dudas sobre si ambos registros corresponden realmente a la misma obra.

In [54]:
# Índices de los registros que se eliminarán
ids_eliminar = [
    5876,  # Ella. Hija de la sabiduría (diferencia solo de puntuación)
    1803,  # Ben-Hur (Traducción Raimundo Grinó)
    8462,  # Príncipe y mendigo
    739,   # Comentarios psicológicos sobre las enseñanzas...
    8032,  # Manual para viajeros por Andalucía y lectores... (duplicado exacto)
    7456,  # Contrato social
    8733,  # El hombre que fue Jueves (trad. José Rafael...)
    5290   # La emperatriz de los Etéreos (Ed. Ilustrada)
]

df = df.drop(index=ids_eliminar).reset_index(drop=True)

print(f"Registros eliminados: {len(ids_eliminar)}")
print(f"Total de libros luego de la limpieza: {len(df)}")

Registros eliminados: 8
Total de libros luego de la limpieza: 8640


# PASO 2

## 2.1

In [41]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, confusion_matrix

df_muestra = pd.read_csv("muestra_etiquetada.csv", keep_default_na=False)
desarrollo, prueba = train_test_split(df_muestra, test_size=0.3, random_state=42, stratify=df_muestra["contaminado"])
X_train, y_train = desarrollo["synopsis"], desarrollo["contaminado"]
X_test,  y_test  = prueba["synopsis"],     prueba["contaminado"]

print(f"Tamaño del conjunto de desarrollo: {len(desarrollo)}")
print(f"Tamaño del conjunto de prueba: {len(prueba)}")

Tamaño del conjunto de desarrollo: 350
Tamaño del conjunto de prueba: 150


In [42]:
df_muestra.head()

,slug,title,synopsis,contaminado,fragmento
0,el-cinturon-de-venus,El cinturón de Venus,"La parábola, la intriga y el humor, tan escaso...",0,
1,los-hijos-de-la-llanura-2,Los hijos de la llanura,Una historia apasionante que te atrapará desde...,1,Una historia apasionante que te atrapará desde...
2,electra,Electra,Entre las siete tragedias de Sófocles (c. 496 ...,0,
3,el-despertar-de-la-senorita-prim,El despertar de la señorita Prim,Atraída por un sugestivo anuncio en un periódi...,0,
4,comprometer-a-un-vizconde,Comprometer a un Vizconde,"Su vida es el ejército, no tiene tiempo para e...",0,


In [43]:
# Celda de código: Expresión regular y evaluación
# Regex para URLs (http/https o www), Correos y Teléfonos (formatos internacionales y locales básicos)
patron_contaminacion = re.compile(
    r'(https?://\S+|www\.\S+)' # URLs
    r'|([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})' # Correos
    r'|(\+?\d[\d\s\-\(\)]{7,15}\d)', # Teléfonos
    re.IGNORECASE
)

# Función para predecir si está contaminado basándose en la regex
def predecir_con_regex(texto):
    if patron_contaminacion.search(str(texto)):
        return 1
    return 0

# Aplicar el detector al conjunto de prueba
y_pred = X_test.apply(predecir_con_regex)

# Medir Precisión y Recall
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"Precisión (Precision): {precision:.4f}")
print(f"Exhaustividad (Recall): {recall:.4f}")

# Explorar falsos positivos
falsos_positivos = X_test[(y_pred == 1) & (y_test == 0)]
print(f"\nCantidad de Falsos Positivos: {len(falsos_positivos)}")
# print(falsos_positivos.head()) # Descomentar para leer ejemplos

Precisión (Precision): 0.6364
Exhaustividad (Recall): 0.2333

Cantidad de Falsos Positivos: 4


## 2.2

In [45]:
import unicodedata
import re
from sklearn.metrics import precision_score, recall_score

def normalizar(texto):
    if not isinstance(texto, str):
        return ""
    
    # 1. Normalizar caracteres de otros alfabetos/anchuras (ej. ｗ -> w, ａ -> a)
    texto = unicodedata.normalize('NFKC', texto)
    
    # 2. Filtrar caracteres invisibles o de control (Categorías Cf y Cc, excepto saltos de línea/tabs)
    texto = "".join(
        c for c in texto 
        if unicodedata.category(c) not in ['Cf', 'Cc'] or c in '\n\r\t'
    )
    
    # 3. Reemplazar palabras usadas para disimular signos
    reemplazos = {
        r'\bpunto\b': '.',
        r'\bdot\b': '.',
        r'\barroba\b': '@',
        r'\bat\b': '@'
    }
    for patron, reemplazo in reemplazos.items():
        texto = re.sub(patron, reemplazo, texto, flags=re.IGNORECASE)
    
    # 4. Eliminar espacios estratégicos en subcadenas típicas de enlaces y correos
    texto = re.sub(r'(?i)\bw\s+w\s+w\b', 'www', texto)
    texto = re.sub(r'(?i)\bc\s+o\s+m\b', 'com', texto)
    
    # 5. Juntar signos despegados (ej. "www . misitio . com" -> "www.misitio.com")
    texto = re.sub(r'\s*\.\s*', '.', texto)
    texto = re.sub(r'\s*@\s*', '@', texto)
    
    return texto

# Aplicar la normalización al conjunto de prueba
X_test_norm = X_test.apply(normalizar)

# Volver a predecir usando la misma regex del Paso 2.1
y_pred_norm = X_test_norm.apply(predecir_con_regex)

# Medir el impacto en las métricas
precision_norm = precision_score(y_test, y_pred_norm)
recall_norm = recall_score(y_test, y_pred_norm)

print(f"Precisión (Normalizado): {precision_norm:.4f}")
print(f"Exhaustividad (Recall Normalizado): {recall_norm:.4f}")

# Identificar los casos que ANTES no se detectaban y AHORA sí (Falsos Negativos -> Verdaderos Positivos)
casos_recuperados = X_test[(y_pred == 0) & (y_pred_norm == 1) & (y_test == 1)]

print(f"\nSe recuperaron {len(casos_recuperados)} registros contaminados ocultos.")
print("\nEjemplos de casos recuperados:")
# Mostrar 2 casos recuperados
for texto_original in casos_recuperados.head(2):
    print(f"\nOriginal : {texto_original}")
    print(f"Normalizado: {normalizar(texto_original)}")

Precisión (Normalizado): 0.6923
Exhaustividad (Recall Normalizado): 0.3000

Se recuperaron 2 registros contaminados ocultos.

Ejemplos de casos recuperados:

Original : Las opiniones del nuevo Papa Francisco I sobre la familia, la fe y el papel de la Iglesia en el siglo XXI. Buscalo en w w w . novelalibre . example El Cardenal Jorge Mario Bergoglio, desde el 13 de marzo de 2013 Papa Francisco, y el Rabino Abraham Skorka, son dos tenaces promotores del diálogo interreligioso, a través del cual buscan construir horizontes comunes sin diluir las particularidades que los caracterizan. Sobre el cielo y la tierra es el resultado de una serie de profundas conversaciones que mantuvieron de manera alternada en la sede del Episcopado y en la comunidad judía Benei Tikva. En sus encuentros transitaron las más variadas cuestiones teológicas y terrenales. Dios, el fundamentalismo, los ateos, la muerte, el Holocausto, la homosexualidad, el capitalismo, son apenas un puñado de los temas en los que dan

## 2.3

In [ ]:
from langdetect import detect, detect_langs, DetectorFactory
DetectorFactory.seed = 42

def detectar_idioma_global(texto):
    try:
        return detect(str(texto))
    except Exception:
        return "unknown"

idiomas_globales = desarrollo["synopsis"].apply(detectar_idioma_global)
print(pd.crosstab(idiomas_globales, desarrollo["contaminado"]))

In [50]:
def oraciones(texto):
    partes = re.split(r"(?<=[.!?])\s+|\n", texto)
    return [p.strip() for p in partes if len(p.strip()) >= 15]

def info_oraciones(texto):
    """Para cada oración: (cantidad de palabras, idioma más probable, su probabilidad)."""
    out = []
    for o in oraciones(texto):
        try:
            l = detect_langs(o)[0]
            out.append((len(o.split()), l.lang, l.prob))
        except Exception:
            pass
    return out

def marcar(info, min_pal, umbral):
    return int(any(n >= min_pal and lang != "es" and p >= umbral for n, lang, p in info))

In [51]:
info_dev = desarrollo["synopsis"].apply(info_oraciones)

filas = []
for mp in [3, 4, 5, 6, 8, 10]:
    for um in [0.7, 0.8, 0.9, 0.95, 0.99]:
        pred = info_dev.apply(lambda i: marcar(i, mp, um))
        filas.append((mp, um,
                      precision_score(desarrollo["contaminado"], pred, zero_division=0),
                      recall_score(desarrollo["contaminado"], pred)))

tabla = pd.DataFrame(filas, columns=["min_pal", "umbral", "precision", "recall"])
display(tabla.sort_values(["precision", "recall"], ascending=False).head(15))

,min_pal,umbral,precision,recall
22,8,0.90,0.818182,0.128571
23,8,0.95,0.818182,0.128571
24,8,0.99,0.818182,0.128571
27,10,0.90,0.818182,0.128571
28,10,0.95,0.818182,0.128571
29,10,0.99,0.818182,0.128571
20,8,0.70,0.733333,0.157143
21,8,0.80,0.733333,0.157143
25,10,0.70,0.733333,0.157143
26,10,0.80,0.733333,0.157143


In [52]:
MIN_PAL, UMBRAL = 5, 0.95   # cambialos por los que elegiste en la celda 3

info_test = prueba["synopsis"].apply(info_oraciones)
y_pred_idioma = info_test.apply(lambda i: marcar(i, MIN_PAL, UMBRAL))

precision_idioma = precision_score(prueba["contaminado"], y_pred_idioma, zero_division=0)
recall_idioma = recall_score(prueba["contaminado"], y_pred_idioma)
print(f"Idioma por oración: precisión {precision_idioma:.2f}, recall {recall_idioma:.2f}")

Idioma por oración: precisión 0.54, recall 0.43


## 2.4

In [53]:
# !pip install gliner
from gliner import GLiNER
from collections import Counter
import random

modelo_gliner = GLiNER.from_pretrained("urchade/gliner_multi_pii-v1")
etiquetas = ["persona", "organización", "lugar", "sitio web", "marca comercial", "precio", "correo electrónico"]

# Oraciones de los fragmentos contaminados (desarrollo) y la misma cantidad de oraciones limpias
frag = desarrollo[desarrollo["contaminado"] == 1]["fragmento"]
oraciones_contaminadas = [o for f in frag for o in (oraciones(f) or [f])]
pool = [o for s in desarrollo[desarrollo["contaminado"] == 0]["synopsis"] for o in oraciones(s)]
oraciones_limpias = random.Random(42).sample(pool, len(oraciones_contaminadas))

def extraer_entidades(textos, modelo, etiquetas):
    resultados = []
    for texto in textos:
        for ent in modelo.predict_entities(texto, etiquetas, threshold=0.5):
            resultados.append((ent["text"], ent["label"]))
    return resultados

entidades_contaminadas = extraer_entidades(oraciones_contaminadas, modelo_gliner, etiquetas)
entidades_limpias = extraer_entidades(oraciones_limpias, modelo_gliner, etiquetas)

print("Etiquetas en CONTAMINADOS")
print(Counter(e for _, e in entidades_contaminadas))
print("\nEtiquetas en LIMPIOS")
print(Counter(e for _, e in entidades_limpias))

print("\nEjemplos CONTAMINADOS")
for texto, etiqueta in list(set(entidades_contaminadas))[:15]:
    print(f"[{etiqueta.upper()}] {texto}")

c:\Users\emmam\Documents\Procesamiento-del-Lenguaje-Natural\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Etiquetas en CONTAMINADOS
Counter({'sitio web': 16, 'organización': 16, 'lugar': 5})

Etiquetas en LIMPIOS
Counter({'persona': 49, 'lugar': 25, 'organización': 18, 'sitio web': 1, 'marca comercial': 1})

Ejemplos CONTAMINADOS
[SITIO WEB] http://www.letraplus.test
[ORGANIZACIÓN] LibroLibre
[SITIO WEB] librоexpress.invalid
[SITIO WEB] http://www.lectorexpress.test
[SITIO WEB] oplus.example
[SITIO WEB] lеctorflаsh.test
[ORGANIZACIÓN] escuela-de-ajedrez
[SITIO WEB] www.epubya.invalid
[ORGANIZACIÓN] LetraLibre
[ORGANIZACIÓN] TomoHub
[ORGANIZACIÓN] LectorPlus
[ORGANIZACIÓN] EpubMax
[ORGANIZACIÓN] LetraTotal
[SITIO WEB] http://www.relatoplus.invalid
[SITIO WEB] librohub.test


## 2.5

#### Construcción del conjunto de oraciones

In [74]:
from nltk.tokenize import sent_tokenize

datos_oraciones = []

# Positivos
for frag in desarrollo["fragmento"].dropna():

    frag = str(frag).strip()

    if frag != "":
        datos_oraciones.append({
            "texto": frag,
            "contaminado": 1
        })


# Negativos
for sinopsis in desarrollo.loc[
    desarrollo["contaminado"] == 0,
    "synopsis"
]:

    for oracion in sent_tokenize(str(sinopsis), language="spanish"):

        oracion = oracion.strip()

        if len(oracion.split()) >= 4:

            datos_oraciones.append({
                "texto": oracion,
                "contaminado": 0
            })

oraciones_df = pd.DataFrame(datos_oraciones)
print(oraciones_df["contaminado"].value_counts())
display(oraciones_df.head())

contaminado
0    1711
1      70
Name: count, dtype: int64


,texto,contaminado
0,El libro más esperado del año llega por fin en...,1
1,"Download this ebook for FREE, no sign-up requi...",1
2,Una historia apasionante que te atrapará desde...,1
3,"Baixe este livro grátis agora mesmo, sem cadas...",1
4,Conseguí este título y más de 50.000 libros en...,1


#### Split

In [75]:
from sklearn.model_selection import train_test_split

X_train_sent, X_val_sent, y_train_sent, y_val_sent = train_test_split(
    oraciones_df["texto"],
    oraciones_df["contaminado"],
    test_size=0.25,
    random_state=42,
    stratify=oraciones_df["contaminado"]
)

print(len(X_train_sent), len(X_val_sent))

1335 446


#### Entrenamiento del clasificador TF-IDF

In [77]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

modelo_tfidf = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1,2),
            min_df=2
        )
    ),
    (
        "clf",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

modelo_tfidf.fit(X_train_sent, y_train_sent)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


#### Selección del umbral de decisión

In [82]:
from sklearn.metrics import precision_recall_curve
import pandas as pd

probs = modelo_tfidf.predict_proba(X_val_sent)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_val_sent,
    probs
)

curva = pd.DataFrame({
    "umbral": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1]
})

curva[curva["precision"] >= 0.90].head(10)

,umbral,precision,recall
425,0.479968,0.944444,0.944444
426,0.541810,1.000000,0.944444
427,0.609327,1.000000,0.888889
428,0.789295,1.000000,0.833333
429,0.865178,1.000000,0.777778
430,0.883076,1.000000,0.722222
431,0.890232,1.000000,0.666667
432,0.893101,1.000000,0.611111
433,0.896817,1.000000,0.555556
434,0.903937,1.000000,0.500000


Se selecciona como umbral 0.541810

#### Evaluación sobre el conjunto de prueba

In [83]:
UMBRAL = 0.54

def detectar_promocion(sinopsis):

    oraciones = sent_tokenize(str(sinopsis), language="spanish")
    if len(oraciones) == 0:
        return 0
    probs = modelo_tfidf.predict_proba(oraciones)[:,1]
    return int(probs.max() >= UMBRAL)

y_pred = prueba["synopsis"].apply(detectar_promocion)

precision = precision_score(prueba["contaminado"], y_pred, zero_division=0)
recall = recall_score(prueba["contaminado"], y_pred)

print(f"Precisión: {precision:.4f}")
print(f"Recall: {recall:.4f}")

Precisión: 1.0000
Recall: 0.8333


### Resultados

Con el umbral seleccionado de 0.54, el clasificador basado en TF-IDF obtiene una precisión del 100 %, lo que indica que no produjo falsos positivos sobre el conjunto de prueba. Además, alcanzó un recall del 83.3 %, detectando la mayoría de las sinopsis contaminadas.

Estos resultados muestran que la representación TF-IDF permite capturar patrones léxicos característicos del texto promocional, aunque algunos casos continúan sin ser detectados.